# synthpose-webots — Geração de dataset COCO-pose do NAO no Kaggle

Este notebook roda o pipeline de geração de dataset sintético (Blender 5.1.2 headless + Cycles GPU)
diretamente em um **Kaggle Notebook**.

## Antes de rodar (menu à direita do Kaggle)
1. **Settings → Accelerator → GPU** (T4 x2 ou P100). Necessário para o Cycles renderizar rápido.
2. **Settings → Internet → On**. Necessário para baixar o Blender e clonar o repositório.

## O que o notebook faz
1. Baixa e extrai o **Blender 5.1.2** (Linux x64).
2. Clona o repositório `synthpose-webots`.
3. Instala `pyyaml` no Python embutido do Blender (numpy já vem embutido; `cv2`/`pycocotools`
   só são usados na validação, que roda no Python do Kaggle).
4. Habilita a **GPU no Cycles** (OptiX → CUDA) via um wrapper.
5. Roda `dataset_generator_phobos.py` gravando em `/kaggle/working/output`.
6. Valida o JSON COCO e mostra uma amostra renderizada.
7. Compacta a saída para download.


## 1. Configuração

Defina o **job** na célula abaixo: `TOTAL_SAMPLES` (total do dataset inteiro),
`WORLD_SIZE` (quantas máquinas Kaggle) e `RANK` (índice desta máquina, de 0).

Cada máquina pega uma **fatia disjunta** (`shard_range`) e, dentro dela, divide o
trabalho entre as **GPUs locais** — 1 processo Blender por T4 (a T4 x2 do Kaggle
gera em paralelo). Nomes de arquivo e sementes nunca colidem entre máquinas nem
entre GPUs, então o merge no fim só renumera os IDs COCO.

**2 máquinas:** mesmo `TOTAL_SAMPLES`/`WORLD_SIZE=2`, muda só `RANK` (0 e 1).
**1 máquina de 2 T4:** deixe `WORLD_SIZE=1, RANK=0` — as duas GPUs já são usadas.
O Kaggle limita a sessão a ~9h; comece com `TOTAL_SAMPLES` pequeno (ex.: 40) para
validar antes do lote grande.

> **Repositório privado?** Se `git clone` falhar por autenticação, faça upload do repo como um
> *Kaggle Dataset* e aponte `REPO_DIR` para `/kaggle/input/<seu-dataset>/synthpose-webots`
> (pule a célula de clone).

In [ ]:
import os

# --- Blender ---
BLENDER_VERSION = "5.1.2"
BLENDER_SERIES  = "Blender5.1"
BLENDER_URL = f"https://download.blender.org/release/{BLENDER_SERIES}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

# --- Repositorio ---
REPO_URL    = "https://github.com/Rinaldots/synthpose-webots.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/synthpose-webots"   # ajuste p/ /kaggle/input/... se usar Dataset

# --- Job multi-maquina (orquestracao) ---
# Declare o job UMA vez e mude apenas RANK em cada maquina Kaggle:
#   Maquina 0 -> RANK=0 ;  Maquina 1 -> RANK=1 ;  ...
# O gerador deriva --start/--num de uma fatia DISJUNTA (ver nao_coco_pose.sharding),
# entao os nomes de arquivo e as sementes nunca colidem entre maquinas.
TOTAL_SAMPLES = 10000   # total do dataset inteiro, somando TODAS as maquinas
WORLD_SIZE    = 2       # quantas maquinas Kaggle vao rodar
RANK          = 0       # <<< MUDE POR MAQUINA: 0 na primeira, 1 na segunda, ...

OUT_DIR = "/kaggle/working/output"    # persiste e fica disponivel p/ download
USE_GPU = True

# --- Desempenho ---
# PROCS_PER_GPU: quantos processos Blender por GPU. Como a GPU fica ociosa durante
# o setup de cena (CPU), rodar 2 por GPU sobrepoe o setup de um com o render de
# outro -> mais throughput. 2 costuma ser o ponto doce nos ~4 vCPU do Kaggle.
PROCS_PER_GPU = 2
# SAVE_EVERY: grava o JSON COCO a cada N imagens (checkpoint). =1 reescreve o JSON
# INTEIRO a cada frame (fica O(N^2) conforme cresce). 100 mantem retomada barata.
SAVE_EVERY = 100

# --- Monitoramento remoto (opcional, via ntfy.sh) ---
# Deixe "" para desativar. Com um topico setado, cada maquina PUBLICA seu status
# (GPU + progresso) e voce agrega tudo no seu PC com:
#   python dataset_generator/scripts/monitor.py --subscribe <topico> --world-size 2
# Use um nome LONGO e ALEATORIO: funciona como senha (quem souber o topico, le).
NTFY_TOPIC = "synthpose-rinaldo-9f3a2b7c1d"   # ex.: "synthpose-rinaldo-9f3a2b7c1d"

# --- Backup incremental do dataset pro seu PC (opcional, via rclone -> nuvem) ---
# Deixe "" para desativar. Com um destino setado, o monitor faz `rclone copy` do
# OUT_DIR a cada BACKUP_EVERY imagens (incremental: sobe so os arquivos NOVOS).
# No seu PC, sincronize a mesma pasta (Google Drive Desktop, ou:
#   rclone copy gdrive:synthpose-backup ./output_backup). O setup do rclone e a
# config (RCLONE_CONF) vao na proxima celula.
BACKUP_REMOTE = ""    # ex.: "gdrive:synthpose-backup"
BACKUP_EVERY  = 100
# Cole aqui o bloco do seu rclone.conf (gerado no PC com `rclone config`). Ex.:
#   [gdrive]
#   type = drive
#   scope = drive
#   token = {"access_token":"...","refresh_token":"...",...}
# Deixe "" se BACKUP_REMOTE="" ou se ja configurou o rclone de outro jeito.
RCLONE_CONF = ""

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
assert 0 <= RANK < WORLD_SIZE, "RANK deve estar em [0, WORLD_SIZE)"
print(f"Config OK  |  job total={TOTAL_SAMPLES}  world_size={WORLD_SIZE}  rank={RANK}"
      f"  procs/gpu={PROCS_PER_GPU}  save_every={SAVE_EVERY}"
      + (f"  ntfy={NTFY_TOPIC}" if NTFY_TOPIC else "  (ntfy off)")
      + (f"  backup->{BACKUP_REMOTE}(/{BACKUP_EVERY})" if BACKUP_REMOTE else "  (backup off)"))

In [ ]:
# --- Setup do rclone p/ backup (roda so se BACKUP_REMOTE estiver setado) ---
# Como configurar UMA vez, no SEU PC:
#   1) instale o rclone e rode `rclone config`
#   2) crie um remote do tipo Google Drive (faz o login no navegador)
#   3) abra ~/.config/rclone/rclone.conf, copie o bloco [<nome>] inteiro
#   4) cole em RCLONE_CONF (celula anterior) e ponha BACKUP_REMOTE="<nome>:synthpose-backup"
import os, shutil, subprocess

def _ensure_rclone():
    if shutil.which("rclone"):
        return True
    print("Instalando rclone via apt...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y rclone 2>/dev/null", shell=True)
    if shutil.which("rclone"):
        return True
    print("apt falhou; tentando script oficial...")
    subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
    return bool(shutil.which("rclone"))

if BACKUP_REMOTE:
    assert _ensure_rclone(), "rclone nao pode ser instalado; backup indisponivel"
    print(subprocess.run(["rclone", "version"], capture_output=True, text=True).stdout.splitlines()[0])
    if RCLONE_CONF.strip():
        cfg_dir = os.path.expanduser("~/.config/rclone")
        os.makedirs(cfg_dir, exist_ok=True)
        with open(os.path.join(cfg_dir, "rclone.conf"), "w") as f:
            f.write(RCLONE_CONF)
        print("rclone.conf escrito em", cfg_dir)
    # Testa o remote. Com scope=drive.file o 'lsd' na raiz pode vir vazio (rc=0) — ok;
    # o rclone so enxerga o que ele mesmo cria, e o 'copy' cria a pasta na 1a vez.
    remote_name = BACKUP_REMOTE.split(":")[0]
    r = subprocess.run(["rclone", "lsd", remote_name + ":"], capture_output=True, text=True)
    print("rclone remote OK" if r.returncode == 0 else f"rclone remote FALHOU:\n{r.stderr[:400]}")
else:
    print("Backup desativado (BACKUP_REMOTE vazio).")

## 2. Baixar e extrair o Blender 5.1.2

In [ ]:
import os, stat, shutil, subprocess, glob, tarfile, urllib.request

TARBALL = f"{WORK}/blender.tar.xz"
BLENDER_HOME = f"{WORK}/blender-{BLENDER_VERSION}-linux-x64"

if not os.path.isdir(BLENDER_HOME):
    # download.blender.org recusa User-Agent nao-navegador (HTTP 403);
    # por isso enviamos um User-Agent de navegador.
    if not os.path.exists(TARBALL) or os.path.getsize(TARBALL) < 1_000_000:
        print("Baixando Blender...", BLENDER_URL)
        req = urllib.request.Request(BLENDER_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(TARBALL, "wb") as f:
            shutil.copyfileobj(r, f)
    print("Extraindo...")
    with tarfile.open(TARBALL) as t:
        t.extractall(WORK)

BLENDER = os.path.join(BLENDER_HOME, "blender")
assert os.path.exists(BLENDER), f"blender nao encontrado em {BLENDER}"

# Restaura o bit de execucao: a persistencia do /kaggle/working entre sessoes pode
# remover +x do binario do Blender e do Python embutido (causa PermissionError).
for exe in [BLENDER] + glob.glob(f"{BLENDER_HOME}/*/python/bin/python3*"):
    os.chmod(exe, os.stat(exe).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

# Bibliotecas de sistema que o Blender 5.1 headless exige no Kaggle.
# libvulkan1 e' obrigatoria: o binario linka libvulkan.so.1 (sem ela: rc=127,
# "error while loading shared libraries: libvulkan.so.1").
subprocess.run("apt-get -qq update && apt-get -qq install -y libvulkan1 libxrender1 libxi6 "
               "libxxf86vm1 libxfixes3 libxkbcommon0 libsm6 libgl1 libegl1 2>/dev/null || true",
               shell=True)

# Sanity check: falha ruidosamente aqui se o Blender ainda nao roda (ex.: lib faltando).
ver = subprocess.run([BLENDER, "--version"], capture_output=True, text=True)
assert ver.returncode == 0, f"Blender nao executa:\n{ver.stderr}"
print(ver.stdout)

## 3. Clonar o repositório
Pule esta célula e ajuste `REPO_DIR` na célula 1 se o repo estiver como *Kaggle Dataset*.

In [ ]:
import os, subprocess

# Clona se ainda nao existe; se JA existe, SINCRONIZA com o remoto (forca hard reset
# p/ pegar as ultimas alteracoes sem risco de conflito). Assim, re-rodar esta celula
# sempre traz a versao mais recente do branch.
if REPO_DIR.startswith("/kaggle/working"):
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH],
                       check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"],
                       check=True)
        print("Repo sincronizado com origin/" + REPO_BRANCH)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                       check=True)
        print("Repo clonado")

DG = os.path.join(REPO_DIR, "dataset_generator")
BLENDER_DIR = os.path.join(DG, "blender")
assert os.path.isfile(os.path.join(BLENDER_DIR, "nao_blender.blend")), \
    f"nao_blender.blend nao encontrado em {BLENDER_DIR}"
print("Repo pronto em:", REPO_DIR)

## 4. Instalar `pyyaml` no Python embutido do Blender
O gerador roda dentro do Blender, então a dependência precisa ir no Python **dele**
(não no do Kaggle). `numpy` já vem embutido.

In [ ]:
import glob, subprocess

BPY = glob.glob(os.path.join(BLENDER_HOME, "*", "python", "bin", "python3*"))
BPY = sorted(BPY)[0]
print("Python do Blender:", BPY)

subprocess.run([BPY, "-m", "ensurepip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "--upgrade", "pip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "pyyaml"], check=True)
print(subprocess.run([BPY, "-c", "import yaml, numpy; print('yaml', yaml.__version__, '| numpy', numpy.__version__)"],
                     capture_output=True, text=True).stdout)

## 5. Wrapper para habilitar a GPU no Cycles
O `dataset_generator_phobos.py` define o engine como CYCLES mas deixa o *device* em CPU.
Este wrapper é passado com um `--python` **antes** do gerador para ligar OptiX/CUDA.

In [ ]:
GPU_WRAPPER = os.path.join(BLENDER_DIR, "kaggle_enable_gpu.py")

with open(GPU_WRAPPER, "w") as f:
    f.write('''
import bpy

prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for dtype in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dtype
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == dtype for d in prefs.devices):
        chosen = dtype
        break

enabled = 0
if chosen:
    for d in prefs.devices:
        d.use = (d.type == chosen)
        enabled += int(d.use)
    for sc in bpy.data.scenes:
        sc.cycles.device = "GPU"

print(f"[KAGGLE-GPU] compute_device_type={chosen} devices_enabled={enabled}")
''')

print("Wrapper escrito em", GPU_WRAPPER)

## 6. Gerar (N processos por GPU) + monitor
Pega a fatia desta máquina (`shard_range`) e a divide entre `n_gpu × PROCS_PER_GPU`
processos Blender — cada um preso a uma GPU (`CUDA_VISIBLE_DEVICES`), gravando em
`output/g{gpu}_p{k}`. Rodar >1 processo por GPU **sobrepõe o setup de cena (CPU) de
um com o render (GPU) de outro**, subindo o throughput quando a GPU fica ociosa.
`--resume` pula PNGs já feitos (retomável); `--save-every` evita reescrever o COCO
inteiro a cada frame. Um **monitor** mostra a cada 15s as GPUs (`nvidia-smi`) + o
progresso somado; com `NTFY_TOPIC`, publica p/ o painel no seu PC.

In [ ]:
import subprocess, os, sys, threading

sys.path.insert(0, os.path.join(DG, "scripts"))
sys.path.insert(0, os.path.join(DG, "src"))
from monitor import watch                       # count_done soma todos os output/g*_p*/images
from nao_coco_pose.sharding import shard_range

# Fatia desta MAQUINA (rank) dentro do job inteiro.
mshard = shard_range(TOTAL_SAMPLES, WORLD_SIZE, RANK)
M_START, M_NUM = mshard.start, mshard.num
print(f"Maquina rank {RANK}/{WORLD_SIZE}: start={M_START} num={M_NUM}")

n_gpu = max(1, subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("UUID"))
n_proc = n_gpu * PROCS_PER_GPU
print(f"GPUs: {n_gpu}  |  processos: {n_proc} ({PROCS_PER_GPU}/GPU)")

# Divide a fatia da maquina entre TODOS os processos (contiguo, a partir de M_START).
# Cada processo fixa 1 GPU (CUDA_VISIBLE_DEVICES) e escreve em output/g{gpu}_p{k}.
base, rem = divmod(M_NUM, n_proc)
shards, cursor, idx = [], M_START, 0
for g in range(n_gpu):
    for k in range(PROCS_PER_GPU):
        num = base + (1 if idx < rem else 0)
        shards.append({"gpu": str(g), "start": cursor, "num": num,
                       "out": f"{OUT_DIR}/g{g}_p{k}"})
        cursor += num; idx += 1

os.makedirs(OUT_DIR, exist_ok=True)
procs = []
for sh in shards:
    if sh["num"] <= 0:
        continue
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=sh["gpu"])
    cmd = [BLENDER, "--background", "nao_blender.blend"]
    if USE_GPU:
        cmd += ["--python", "kaggle_enable_gpu.py"]
    # --resume: pula PNGs ja feitos (retomavel sem duplicar). --save-every: checkpoint
    # do JSON a cada N (evita reescrever o COCO inteiro a cada frame).
    cmd += ["--python", "dataset_generator_phobos.py",
            "--", "--num", str(sh["num"]), "--start", str(sh["start"]),
            "--out", sh["out"], "--resume", "--save-every", str(SAVE_EVERY)]
    log = open(sh["out"] + ".log", "w")
    sh["logpath"] = log.name
    print(f"GPU {sh['gpu']} -> {os.path.basename(sh['out'])}: start={sh['start']} num={sh['num']}")
    procs.append((sh, subprocess.Popen(cmd, cwd=BLENDER_DIR, env=env,
                                       stdout=log, stderr=subprocess.STDOUT)))

# Monitor: 1 thread p/ a maquina toda — mostra as GPUs + progresso somado, publica
# no ntfy (se NTFY_TOPIC) e faz backup incremental via rclone (se BACKUP_REMOTE).
stop = threading.Event()
mon = threading.Thread(target=watch, kwargs=dict(
        out_dir=OUT_DIR, num=M_NUM, interval=15.0, stop=stop,
        rank=RANK, ntfy_topic=(NTFY_TOPIC or None),
        backup_remote=(BACKUP_REMOTE or None), backup_every=BACKUP_EVERY), daemon=True)
mon.start()
try:
    for sh, p in procs:
        print(f"{os.path.basename(sh['out'])} terminou  rc={p.wait()}")
finally:
    stop.set(); mon.join(timeout=17)
    for sh, _ in procs:                          # tail do log de cada processo
        with open(sh["logpath"]) as f:
            tail = f.readlines()[-2:]
        print(f"--- {os.path.basename(sh['out'])} (fim do log) ---\n{''.join(tail)}")

## 7. Validar o JSON COCO e visualizar uma amostra

In [ ]:
import glob, subprocess, json

# Valida o JSON COCO de cada GPU (roda no Python do Kaggle; pycocotools ja disponivel).
# Busca recursiva cobre output/gpu*/annotations/ (1 processo por GPU).
val = os.path.join(DG, "scripts", "validate_dataset.py")
ann = sorted(glob.glob(os.path.join(OUT_DIR, "**", "person_keypoints_*.json"), recursive=True))
env = dict(os.environ, PYTHONPATH=os.path.join(DG, "src"))
for a in ann:
    print("===", os.path.relpath(a, OUT_DIR), "===")
    subprocess.run(["python", val, a], env=env)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Busca recursiva cobre output/gpu*/images/** (1 processo por GPU).
imgs = sorted(glob.glob(os.path.join(OUT_DIR, "**", "images", "**", "*.png"), recursive=True))
print(f"{len(imgs)} imagens geradas")
if imgs:
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = [axes] if n == 1 else axes
    for ax, p in zip(axes, imgs[:n]):
        ax.imshow(mpimg.imread(p)); ax.set_title(os.path.basename(p)); ax.axis("off")
    plt.tight_layout(); plt.show()

## 8. Compactar a saída para download
O `.zip` aparece na aba **Output** / **Data** do Kaggle para download.

In [ ]:
import shutil
# Nomeia o zip por RANK para os downloads das varias maquinas nao colidirem no seu PC.
zip_base = f"/kaggle/working/nao_dataset_rank{RANK}of{WORLD_SIZE}"
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Pronto:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
print(f"\nBaixe este zip de CADA maquina, depois junte tudo localmente com:")
print(f"  python scripts/merge_datasets.py nao_dataset_rank*.zip --out output_merged")